# HDB resale — CatBoost v1

**Model:** `CatBoostRegressor` on **`log1p(resale_price)`**; metrics and submission use **`expm1`** (dollar scale).

**Validation (two runs):**
1. **Time-based:** hold out the **last 12 calendar months** (`Tranc_YearMonth`).
2. **Random:** **70% / 30%** `train_test_split` (`shuffle=True`, `random_state=RNG`).

**Final submission:** refit on **full** training data using **`tree_count_`** from the **time-based** model (trees learned with early stopping).

**Dependencies:** `pip install catboost` or `conda install -c conda-forge catboost`.

**Submissions:**
- `ROOT / submission / sub_cat_v1_t.csv` (time split)
- `ROOT / submission / sub_cat_v1_r.csv` (random split)



In [8]:
# Paths
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebook" else _cwd
TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUB_PATH = ROOT / "data" / "sample_sub_reg.csv"
SUBMISSION_PATH_T = ROOT / "submission" / "sub_cat_v1_t.csv"
SUBMISSION_PATH_R = ROOT / "submission" / "sub_cat_v1_r.csv"

RNG = 42

print(f"ROOT: {ROOT.resolve()}")
train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
print(train.shape, test.shape)



ROOT: /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle
(150634, 77) (16735, 76)


In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

from catboost import CatBoostRegressor

TARGET = "resale_price"



In [ ]:
ROOMS_FROM_FLAT = {
    "1 ROOM": 1,
    "2 ROOM": 2,
    "3 ROOM": 3,
    "4 ROOM": 4,
    "5 ROOM": 5,
    "EXECUTIVE": 6,
    "MULTI-GENERATION": 7,
}


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    pc = out["postal"].astype(str).str.replace(r"\.0$", "", regex=True)
    out["postal_sector"] = pd.to_numeric(pc.str.slice(0, 2), errors="coerce")

    ms = pd.to_numeric(out["mid_storey"], errors="coerce")
    mx = pd.to_numeric(out["max_floor_lvl"], errors="coerce")
    out["storey_ratio"] = np.where(mx > 0, ms / mx, np.nan)

    rcols = ["1room_rental", "2room_rental", "3room_rental", "other_room_rental"]
    total_rent = np.zeros(len(out))
    for c in rcols:
        total_rent += pd.to_numeric(out[c], errors="coerce").fillna(0).to_numpy(dtype=float)
    td = pd.to_numeric(out["total_dwelling_units"], errors="coerce").to_numpy(dtype=float)
    out["rental_ratio"] = np.where(td > 0, total_rent / td, np.nan)

    out["rooms_num"] = out["flat_type"].map(ROOMS_FROM_FLAT).astype(float)
    ty = pd.to_numeric(out["Tranc_Year"], errors="coerce")
    tm = pd.to_numeric(out["Tranc_Month"], errors="coerce")
    out["month_index"] = (ty - 2000) * 12 + tm
    return out


train = add_engineered_features(train)
test = add_engineered_features(test)

DROP_FEATURES = [
    "id",
    "Tranc_YearMonth",
    "Tranc_Year",
    "Tranc_Month",
    "floor_area_sqft",
    "postal",
    "address",
    "block",
    "street_name",
    "flat_type",
    "flat_model",
    "1room_sold",
    "2room_sold",
    "3room_sold",
    "4room_sold",
    "5room_sold",
    "exec_sold",
    "multigen_sold",
    "studio_apartment_sold",
    "1room_rental",
    "2room_rental",
    "3room_rental",
    "other_room_rental",
    "bus_stop_name",
    "sec_sch_name",
]

feature_cols = [c for c in train.columns if c not in DROP_FEATURES and c != TARGET]
assert not set(feature_cols) - set(test.columns)

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train[TARGET].astype(float)

# For time-based split (full train still has Tranc_YearMonth in `train`)
period = pd.to_datetime(train["Tranc_YearMonth"], format="%Y-%m")

print(
    f"Features: {len(feature_cols)} (incl. month_index from Tranc_Year/Tranc_Month)"
)



Features: 58 (incl. month_index from Tranc_Year/Tranc_Month)


In [11]:
def imputation_stats(X_ref: pd.DataFrame, cat_cols: list, num_cols: list):
    num_med = {
        c: pd.to_numeric(X_ref[c], errors="coerce").median()
        for c in num_cols
    }
    cat_fill = {}
    for c in cat_cols:
        s = X_ref[c].astype(str).replace("nan", np.nan)
        m = s.mode(dropna=True)
        cat_fill[c] = m.iloc[0] if len(m) else "_MISSING_"
    return num_med, cat_fill


def prepare_catboost(
    X: pd.DataFrame,
    cat_cols: list,
    num_cols: list,
    num_med: dict,
    cat_fill: dict,
) -> pd.DataFrame:
    """Numeric medians + categorical string fill; CatBoost uses cat_features by name."""
    out = X.copy()
    for c in num_cols:
        v = pd.to_numeric(out[c], errors="coerce")
        out[c] = v.fillna(num_med[c]).astype(float)
    for c in cat_cols:
        s = out[c].astype(str).replace("nan", np.nan).fillna(cat_fill[c])
        out[c] = s.astype(str)
    return out


num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]
print(f"Numeric: {len(num_cols)}, categorical (CatBoost): {len(cat_cols)}")



Numeric: 47, categorical (CatBoost): 11


In [12]:
# --- A) Time-based validation (last 12 months) ---
cutoff = period.max() - pd.DateOffset(months=12)
tr_time = period < cutoff
va_time = ~tr_time

X_tr_raw = X_train.loc[tr_time]
X_va_raw = X_train.loc[va_time]
y_tr_t = y.loc[tr_time]
y_va_t = y.loc[va_time]

num_med_t, cat_fill_t = imputation_stats(X_tr_raw, cat_cols, num_cols)
X_tr_t = prepare_catboost(X_tr_raw, cat_cols, num_cols, num_med_t, cat_fill_t)
X_va_t = prepare_catboost(X_va_raw, cat_cols, num_cols, num_med_t, cat_fill_t)

y_tr_log_t = np.log1p(y_tr_t.values)
y_va_log_t = np.log1p(y_va_t.values)

model_time = CatBoostRegressor(
    loss_function="RMSE",
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    random_seed=RNG,
    verbose=False,
    early_stopping_rounds=80,
)
model_time.fit(
    X_tr_t,
    y_tr_log_t,
    eval_set=(X_va_t, y_va_log_t),
    cat_features=cat_cols,
)

pred_va_t = np.expm1(model_time.predict(X_va_t))
rmse_t = root_mean_squared_error(y_va_t, pred_va_t)
mae_t = mean_absolute_error(y_va_t, pred_va_t)
print(
    f"Time split: train {len(X_tr_t):,}, val {len(X_va_t):,} (val >= {cutoff.date()})"
)
print(f"Time-split validation RMSE (dollars): {rmse_t:,.2f}")
print(f"Time-split validation MAE (dollars): {mae_t:,.2f}")
print(f"time model tree_count_: {model_time.tree_count_}")



Time split: train 128,899, val 21,735 (val >= 2020-04-01)
Time-split validation RMSE (dollars): 33,097.43
Time-split validation MAE (dollars): 24,835.55
time model tree_count_: 1989


In [ ]:
# --- B) Random 80% / 20% validation ---
X_tr_raw_r, X_va_raw_r, y_tr_r, y_va_r = train_test_split(
    X_train,
    y,
    test_size=0.80,
    random_state=RNG,
    shuffle=True,
)

num_med_r, cat_fill_r = imputation_stats(X_tr_raw_r, cat_cols, num_cols)
X_tr_r = prepare_catboost(X_tr_raw_r, cat_cols, num_cols, num_med_r, cat_fill_r)
X_va_r = prepare_catboost(X_va_raw_r, cat_cols, num_cols, num_med_r, cat_fill_r)

y_tr_log_r = np.log1p(y_tr_r.values)
y_va_log_r = np.log1p(y_va_r.values)

model_rand = CatBoostRegressor(
    loss_function="RMSE",
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    random_seed=RNG,
    verbose=False,
    early_stopping_rounds=80,
)
model_rand.fit(
    X_tr_r,
    y_tr_log_r,
    eval_set=(X_va_r, y_va_log_r),
    cat_features=cat_cols,
)

pred_va_r = np.expm1(model_rand.predict(X_va_r))
rmse_r = root_mean_squared_error(y_va_r, pred_va_r)
mae_r = mean_absolute_error(y_va_r, pred_va_r)
print(
    f"Random 80/20: train {len(X_tr_r):,} ({100 * len(X_tr_r) / len(X_train):.1f}%), "
    f"val {len(X_va_r):,} ({100 * len(X_va_r) / len(X_train):.1f}%)"
)
print(f"Random-split validation RMSE (dollars): {rmse_r:,.2f}")
print(f"Random-split validation MAE (dollars): {mae_r:,.2f}")
print(f"random model tree_count_: {model_rand.tree_count_}")



Random 80/20: train 37,658 (25.0%), val 112,976 (75.0%)
Random-split validation RMSE (dollars): 23,607.86
Random-split validation MAE (dollars): 16,959.49
random model tree_count_: 2000


In [14]:
# --- C) Full train refit for submissions (_t and _r) ---
n_trees_t = int(model_time.tree_count_)
if n_trees_t < 50:
    n_trees_t = 500

n_trees_r = int(model_rand.tree_count_)
if n_trees_r < 50:
    n_trees_r = 500

num_med_f, cat_fill_f = imputation_stats(X_train, cat_cols, num_cols)
X_full = prepare_catboost(X_train, cat_cols, num_cols, num_med_f, cat_fill_f)
X_test_p = prepare_catboost(X_test, cat_cols, num_cols, num_med_f, cat_fill_f)
y_log_full = np.log1p(y.values)

model_final_t = CatBoostRegressor(
    loss_function="RMSE",
    iterations=n_trees_t,
    learning_rate=0.05,
    depth=8,
    random_seed=RNG,
    verbose=False,
)
model_final_r = CatBoostRegressor(
    loss_function="RMSE",
    iterations=n_trees_r,
    learning_rate=0.05,
    depth=8,
    random_seed=RNG,
    verbose=False,
)
model_final_t.fit(X_full, y_log_full, cat_features=cat_cols)
model_final_r.fit(X_full, y_log_full, cat_features=cat_cols)

test_pred_t = np.expm1(model_final_t.predict(X_test_p))
test_pred_r = np.expm1(model_final_r.predict(X_test_p))

sample = pd.read_csv(SAMPLE_SUB_PATH, nrows=5)
sub_t = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_t})
sub_r = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_r})
assert list(sub_t.columns) == list(sample.columns)
assert list(sub_r.columns) == list(sample.columns)

SUBMISSION_PATH_T.parent.mkdir(parents=True, exist_ok=True)
sub_t.to_csv(SUBMISSION_PATH_T, index=False)
sub_r.to_csv(SUBMISSION_PATH_R, index=False)
print(f"Final time model iterations (from time-split tree_count_): {n_trees_t}")
print(f"Final random model iterations (from random-split tree_count_): {n_trees_r}")
print(f"Wrote {SUBMISSION_PATH_T.resolve()} ({len(sub_t):,} rows)")
print(f"Wrote {SUBMISSION_PATH_R.resolve()} ({len(sub_r):,} rows)")
print(sub_t.head())


Final time model iterations (from time-split tree_count_): 1989
Final random model iterations (from random-split tree_count_): 2000
Wrote /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle/submission/sub_cat_v1_t.csv (16,735 rows)
Wrote /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle/submission/sub_cat_v1_r.csv (16,735 rows)
       Id      Predicted
0  114982  372070.036950
1   95653  454371.225085
2   40303  354630.850641
3  109506  289574.084566
4  100149  422883.051270


## Compare splits

- **Random 70/30** RMSE is often **lower** than the **time-based** RMSE because future transactions leak into the training fold (optimistic).
- **Kaggle / production** alignment is closer to the **time-based** score; the submission model uses **`tree_count_`** from that run to cap complexity.

